In [1]:
import os
import re
import csv
import time
import mimetypes
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from urllib.parse import urlparse

import pandas as pd
import requests

In [2]:
# =========================
# CONFIG
# =========================
INPUT_POSTS_CSV = "Instagram/ig_post_info.csv"  # <-- your parsed posts table
OUT_DIR = Path("Instagram/posts_info/Downloads")                                      # <-- where media files go
OUT_MANIFEST_CSV = "downloads_manifest.csv"

SLEEP_SECONDS = 0.2
TIMEOUT = 60
MAX_RETRIES = 2

In [3]:
# =========================
# Helpers
# =========================
def safe_filename(s: str) -> str:
    s = str(s).strip()
    s = re.sub(r"\s+", "_", s)
    return re.sub(r"[^a-zA-Z0-9._-]", "", s)


def split_urls(s: str) -> List[str]:
    if s is None:
        return []
    s = str(s).strip()
    if not s:
        return []
    # your code joins with ", " so splitting on comma is fine
    parts = [p.strip() for p in s.split(",")]
    return [p for p in parts if p]


def guess_ext_from_url(url: str) -> str:
    """
    Best-effort extension guess from URL path.
    Falls back to .bin if unknown.
    """
    path = urlparse(url).path
    ext = Path(path).suffix.lower()
    if ext and len(ext) <= 5:
        return ext
    return ".bin"


def guess_ext_from_content_type(content_type: Optional[str], fallback: str = ".bin") -> str:
    if not content_type:
        return fallback
    # strip charset
    content_type = content_type.split(";")[0].strip().lower()
    ext = mimetypes.guess_extension(content_type) or fallback
    # common fix: jpeg -> .jpg
    if ext == ".jpe":
        ext = ".jpg"
    return ext

In [4]:
def build_tasks(df: pd.DataFrame) -> List[Dict]:
    """
    Build a download task list with fields:
    post_id, slide_number, url, kind, file_title (without extension)
    """
    tasks: List[Dict] = []

    # normalize expected columns
    for col in ["post_id", "post_type", "carousel_media_urls", "video_url", "cover_url"]:
        if col not in df.columns:
            df[col] = ""

    for _, r in df.iterrows():
        post_id = str(r.get("post_id", "")).strip()
        if not post_id:
            continue

        post_type = str(r.get("post_type", "")).strip()
        carousel_urls = split_urls(r.get("carousel_media_urls", ""))
        video_url = str(r.get("video_url", "") or "").strip()
        cover_url = str(r.get("cover_url", "") or "").strip()

        # 1) Carousel: download every slide URL
        if post_type == "GraphSidecar" and carousel_urls:
            for i, u in enumerate(carousel_urls, start=1):
                tasks.append({
                    "post_id": post_id,
                    "slide_number": i,
                    "url": u,
                    "kind": "carousel_slide",
                    "file_title_base": f"{safe_filename(post_id)}_{i}",
                })

        # 2) Single video/reel: download video_url
        elif post_type == "GraphVideo" and video_url:
            tasks.append({
                "post_id": post_id,
                "slide_number": 1,
                "url": video_url,
                "kind": "video",
                "file_title_base": f"{safe_filename(post_id)}",
            })

        # 3) Single image: download cover_url (or display_url if you prefer)
        else :
            tasks.append({
                "post_id": post_id,
                "slide_number": 1,
                "url": cover_url,
                "kind": "cover_image",
                "file_title_base": f"{safe_filename(post_id)}",
            })

        # If none of the above exist, skip (or you can log)
    return tasks


def download_one(session: requests.Session, url: str, out_path_no_ext: Path) -> Tuple[bool, str]:
    """
    Downloads url to out_path_no_ext + inferred extension.
    Returns (downloaded_ok, final_filepath_str)
    """
    # If URL already has a clear ext, we can use it; otherwise infer from headers.
    url_ext = guess_ext_from_url(url)
    # We'll still prefer header ext if url_ext is .bin (unknown)
    final_path = out_path_no_ext.with_suffix(url_ext)

    # If file already exists, skip
    if final_path.exists() and final_path.stat().st_size > 0:
        return True, str(final_path)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 2):  # total attempts = MAX_RETRIES+1
        try:
            with session.get(url, stream=True, timeout=TIMEOUT) as resp:
                resp.raise_for_status()

                # If URL ext unknown, infer from Content-Type
                if url_ext == ".bin":
                    ext2 = guess_ext_from_content_type(resp.headers.get("Content-Type"), fallback=".bin")
                    final_path = out_path_no_ext.with_suffix(ext2)

                # Skip if already downloaded under inferred ext
                if final_path.exists() and final_path.stat().st_size > 0:
                    return True, str(final_path)

                OUT_DIR.mkdir(parents=True, exist_ok=True)
                tmp_path = final_path.with_suffix(final_path.suffix + ".part")

                with tmp_path.open("wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 256):
                        if chunk:
                            f.write(chunk)

                tmp_path.replace(final_path)
                return True, str(final_path)

        except Exception as e:
            last_err = str(e)
            time.sleep(0.5 * attempt)

    return False, f"ERROR: {last_err}"


def run_downloads(
    input_posts_csv: str = INPUT_POSTS_CSV,
    out_dir: Path = OUT_DIR,
    out_manifest_csv: str = OUT_MANIFEST_CSV,
    sleep_seconds: float = SLEEP_SECONDS,
) -> pd.DataFrame:
    global OUT_DIR
    OUT_DIR = out_dir

    df = pd.read_csv(input_posts_csv)

    # Build tasks
    tasks = build_tasks(df)

    # Prepare manifest rows
    manifest_rows: List[Dict] = []

    with requests.Session() as session:
        # Optional: set a UA to reduce blocks
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (compatible; media-downloader/1.0)"
        })

        for idx, t in enumerate(tasks, start=1):
            post_id = t["post_id"]
            slide_number = t["slide_number"]
            url = t["url"]
            file_base = t["file_title_base"]

            out_path_no_ext = OUT_DIR / file_base

            ok, final_path_or_err = download_one(session, url, out_path_no_ext)

            manifest_rows.append({
                "post_id": post_id,
                "slide_number": slide_number,
                "file_title": Path(final_path_or_err).name if ok else file_base,
                "url": url,
                "downloaded": bool(ok),
                "local_path": final_path_or_err if ok else "",
                "error": "" if ok else final_path_or_err,
            })

            print(f"[{idx}/{len(tasks)}] post_id={post_id} slide={slide_number} downloaded={ok}")
            time.sleep(sleep_seconds)

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(out_manifest_csv, index=False, encoding="utf-8")
    print(f"\nSaved manifest: {out_manifest_csv} | rows={len(manifest_df)}")
    return manifest_df

In [5]:
if __name__ == "__main__":
    # Run downloads + create manifest CSV
    run_downloads(
        input_posts_csv=INPUT_POSTS_CSV,
        out_dir=OUT_DIR,
        out_manifest_csv=OUT_MANIFEST_CSV,
        sleep_seconds=SLEEP_SECONDS,
    )


[1/5254] post_id=3835422615627999905 slide=1 downloaded=True
[2/5254] post_id=3835405492215196230 slide=1 downloaded=True
[3/5254] post_id=3835391197422670634 slide=1 downloaded=True
[4/5254] post_id=3835391197422670634 slide=2 downloaded=True
[5/5254] post_id=3835391197422670634 slide=3 downloaded=True
[6/5254] post_id=3835391197422670634 slide=4 downloaded=True
[7/5254] post_id=3835391197422670634 slide=5 downloaded=True
[8/5254] post_id=3835391197422670634 slide=6 downloaded=True
[9/5254] post_id=3835391197422670634 slide=7 downloaded=True
[10/5254] post_id=3835391197422670634 slide=8 downloaded=True
[11/5254] post_id=3834753934065553649 slide=1 downloaded=True
[12/5254] post_id=3834664334044237488 slide=1 downloaded=True
[13/5254] post_id=3834664334044237488 slide=2 downloaded=True
[14/5254] post_id=3834664334044237488 slide=3 downloaded=True
[15/5254] post_id=3834664334044237488 slide=4 downloaded=True
[16/5254] post_id=3834664334044237488 slide=5 downloaded=True
[17/5254] post_id

KeyboardInterrupt: 